# **Fase 5a · Montagem das entradas**
### *Monta, para cada faixa, o arquivo que o LLM vai receber, a partir das saídas das fases 3 e 4, conforme a seção 3 de `docs/fase5/01_desenho_experimental.md`.*

(Este caderno não usa os áudios, a GPU nem a chave do Gemini. Lê apenas os JSON das fases 3 e 4, já publicados no repositório, e grava os cinco arquivos de entrada em `resultados/fase5/entradas`. Pode ser executado por qualquer pessoa a partir do repositório)

## **1. Preparação**

*Conecta o Drive e define as pastas. As entradas vêm de duas fontes, as transcrições do Whisper nas condições A (sem glossário e sem filtro) e C (com filtro), da fase 3, e as execuções do PANNs e do CLAP sem filtro, da fase 4.*

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json
from collections import Counter
import pandas as pd

PASTA_PROJETO = '/content/drive/MyDrive/escuta-cerrada-poc'
PASTA_FASE3 = os.path.join(PASTA_PROJETO, 'resultados', 'fase3')
PASTA_FASE4 = os.path.join(PASTA_PROJETO, 'resultados', 'fase4')
PASTA_ENTRADAS = os.path.join(PASTA_PROJETO, 'resultados', 'fase5', 'entradas')
os.makedirs(PASTA_ENTRADAS, exist_ok=True)

necessarios = [os.path.join(PASTA_FASE3, f'faixa{n:02d}_{c}.json') for n in range(1, 6) for c in 'AC']
necessarios += [os.path.join(PASTA_FASE4, f'faixa{n:02d}_{m}_sem.json') for n in range(1, 6) for m in ['panns', 'clap']]
faltando = [a for a in necessarios if not os.path.exists(a)]
print(len(necessarios) - len(faltando), 'de', len(necessarios), 'arquivos encontrados', faltando or '')

Mounted at /content/drive
20 de 20 arquivos encontrados 


### 1.1 Leitura

Lê os vinte arquivos e mostra, para cada faixa, a duração, o número de segmentos da condição A, o número de palavras da condição C e o número de janelas do PANNs e do CLAP, que devem ser iguais.

In [2]:
def ler(caminho):
    with open(caminho, encoding='utf-8') as f:
        return json.load(f)

DADOS = {}
for n in range(1, 6):
    a = ler(os.path.join(PASTA_FASE3, f'faixa{n:02d}_A.json'))
    c = ler(os.path.join(PASTA_FASE3, f'faixa{n:02d}_C.json'))
    DADOS[n] = {
        'duracao': a['duracao_audio_s'],
        'segmentos_a': a['segmentos'],
        'palavras_c': [w for s in c['segmentos'] for w in s.get('words', [])],
        'panns': ler(os.path.join(PASTA_FASE4, f'faixa{n:02d}_panns_sem.json'))['janelas'],
        'clap': ler(os.path.join(PASTA_FASE4, f'faixa{n:02d}_clap_sem.json'))['janelas'],
    }

pd.DataFrame([{'faixa': n, 'duração (s)': d['duracao'], 'segmentos A': len(d['segmentos_a']),
               'palavras C': len(d['palavras_c']), 'janelas PANNs': len(d['panns']),
               'janelas CLAP': len(d['clap'])} for n, d in DADOS.items()]).set_index('faixa')

,duração (s),segmentos A,palavras C,janelas PANNs,janelas CLAP
faixa,,,,,
1,335.52,63,438,335,335
2,201.32,40,362,201,201
3,362.06,62,673,362,362
4,64.86,8,66,64,64
5,441.32,23,517,441,441


**Resultado.** Os vinte arquivos foram encontrados, e o PANNs e o CLAP analisaram cada faixa no mesmo número de janelas.

## **2. Regras fixadas no desenho experimental**

*Transcreve para o código os limiares da seção 3 do desenho. Os sinais de “indicação de ausência de fala” e de “rótulo apontado” usam os mesmos critérios das fases 3 e 4.*

In [3]:
LIMIAR_REPETICAO = 3              # segmentos com o mesmo texto
LIMIAR_SEM_FALA = 0.6             # no_speech_prob, critério da fase 3
LIMIAR_LOGPROB = -1.0             # avg_logprob
LIMIAR_DIVERGENCIA = 0.5          # proporção de palavras encontradas na condição C
LIMIAR_PANNS = 0.3                # rótulo apontado, critério da fase 4
PROPORCAO_ONIPRESENTE_PANNS = 0.5 # rótulos acima disso ficam fora dos itens
TOP_CLAP = 3                      # posições consideradas para os atratores
PROPORCAO_ATRATOR_CLAP = 0.25     # descrições acima disso ficam fora dos itens

## **3. Fala**

*Para cada segmento da condição A, calcula os quatro sinais da seção 3. A comparação de textos ignora maiúsculas e pontuação. Na divergência, contam as palavras da condição C cujo tempo tem algum trecho em comum com o intervalo do segmento. Segmentos sem texto não recebem os sinais de repetição e de divergência, mas podem receber os outros dois.*

In [4]:
def palavras(texto):
    """Palavras em minúsculas, sem pontuação."""
    return re.sub(r'[^\w\s]', ' ', texto.lower()).split()


def normaliza(texto):
    """Texto em minúsculas, sem pontuação e com espaços simples."""
    return ' '.join(palavras(texto))


def sinais_fala(segmentos, palavras_c):
    """Lista de sinais de cada segmento da condição A.

    palavras_c: palavras da condição C, cada uma com 'word', 'start' e 'end'."""
    contagem = Counter(normaliza(s['text']) for s in segmentos)
    saida = []
    for s in segmentos:
        sinais = []
        texto = normaliza(s['text'])
        if texto and contagem[texto] >= LIMIAR_REPETICAO:
            sinais.append('repetição')
        if s['no_speech_prob'] > LIMIAR_SEM_FALA:
            sinais.append('indicação de ausência de fala')
        if s['avg_logprob'] < LIMIAR_LOGPROB:
            sinais.append('baixa confiança')
        ps = palavras(s['text'])
        if ps:
            em_c = {p for w in palavras_c if w['start'] < s['end'] and w['end'] > s['start']
                    for p in palavras(w['word'])}
            if sum(p in em_c for p in ps) / len(ps) < LIMIAR_DIVERGENCIA:
                sinais.append('divergência entre transcrições')
        saida.append(sinais)
    return saida


SINAIS = {n: sinais_fala(d['segmentos_a'], d['palavras_c']) for n, d in DADOS.items()}

NOMES_SINAIS = ['repetição', 'indicação de ausência de fala', 'baixa confiança', 'divergência entre transcrições']
pd.DataFrame([{'faixa': n, 'segmentos': len(s), 'com algum sinal': sum(bool(x) for x in s),
               **{nome: sum(nome in x for x in s) for nome in NOMES_SINAIS}}
              for n, s in SINAIS.items()]).set_index('faixa')

,segmentos,com algum sinal,repetição,indicação de ausência de fala,baixa confiança,divergência entre transcrições
faixa,,,,,,
1,63,21,3,0,11,17
2,40,28,0,6,17,22
3,62,9,0,0,8,4
4,8,1,0,0,0,1
5,23,15,15,1,0,14


**Resultado.** Dos 196 segmentos da condição A, 74 receberam ao menos um sinal. Na faixa 05, os 15 segmentos do loop “A CIDADE NO BRASIL” receberam o sinal de repetição, mas apenas um recebeu indicação de ausência de fala e nenhum recebeu baixa confiança. O loop só se torna visível, portanto, quando a transcrição é comparada consigo mesma e com a condição C, e não pelos indicadores do próprio Whisper.

### 3.1 Conferência dos sinais

Mostra os segmentos com repetição e alguns exemplos de divergência, com o texto da condição A e as palavras da condição C no mesmo intervalo, para que se possa ver o que cada sinal registra.

In [5]:
def texto_c(segmento, palavras_c):
    return ' '.join(w['word'].strip() for w in palavras_c
                    if w['start'] < segmento['end'] and w['end'] > segmento['start'])

for n, d in DADOS.items():
    rep = [s for s, x in zip(d['segmentos_a'], SINAIS[n]) if 'repetição' in x]
    if rep:
        textos = sorted({s['text'].strip() for s in rep})
        print(f'Faixa {n:02d} · repetição em {len(rep)} segmentos, de {rep[0]["start"]:.1f} a {rep[-1]["end"]:.1f} s · {textos}')

for n, d in DADOS.items():
    div = [s for s, x in zip(d['segmentos_a'], SINAIS[n]) if 'divergência entre transcrições' in x]
    print(f'\nFaixa {n:02d} · {len(div)} segmentos com divergência · primeiros exemplos')
    for s in div[:3]:
        print(f'   {s["start"]:.1f} a {s["end"]:.1f} s')
        print(f'      A: {s["text"].strip()}')
        print(f'      C: {texto_c(s, d["palavras_c"])}')

Faixa 01 · repetição em 3 segmentos, de 252.2 a 328.3 s · ['Aplausos.']
Faixa 05 · repetição em 15 segmentos, de 25.0 a 389.0 s · ['A CIDADE NO BRASIL']

Faixa 01 · 17 segmentos com divergência · primeiros exemplos
   88.8 a 89.6 s
      A: exigência e desigualdade.
      C: existente
   89.6 a 92.6 s
      A: A questão é que a lei não é existente para salariados e empregadores.
      C: existente para salariados e empregadores.
   146.6 a 148.6 s
      A: A MENSAGEM DO PRODUÇÃO DE UMA MENSAGEM DE RENOVOÇÃO
      C: A massa operária

Faixa 02 · 22 segmentos com divergência · primeiros exemplos
   73.4 a 76.9 s
      A: Tenho medo de engolir a estrela d'alva, mas me pergunto por que eu não vou.
      C: Tenho medo de engolir estrelas altas Mas meu degruso
   77.0 a 83.3 s
      A: Eu me debruço e vou beber o céu, como era fresco o céu da minha terra.
      C: degruso e bobagem são Quem mora preso segue o céu
   104.4 a 106.7 s
      A: O gado invadindo a desfogada.
      C: Cada invadin

**Resultado.** A divergência não indica qual transcrição está correta. Na faixa 02, a condição A escreveu versos plausíveis (“estrela d'alva”, “me debruço”) onde a condição C escreveu “degruso”, enquanto na faixa 01, aos 146 s, a condição A escreveu “A MENSAGEM DO PRODUÇÃO DE UMA MENSAGEM DE RENOVOÇÃO” onde a condição C escreveu “A massa operária”. Na faixa 03, os segmentos de 359 a 362 s, em que a condição A escreveu frases e a condição C não escreveu nada, coincidem com o fim do arquivo, onde a escuta da fase 4 registrou silêncio. Na faixa 01, o Whisper escreveu “Aplausos.” em três segmentos, tomando como fala o nome de um som, e esses segmentos receberam o sinal de repetição.

## **4. Eventos (PANNs)**

*Seleciona os rótulos com probabilidade de pelo menos 0,3 em cada janela e reúne as janelas consecutivas do mesmo rótulo num intervalo, com a probabilidade máxima. Os rótulos apontados em mais da metade das janelas da faixa ficam fora dos itens e entram numa linha de resumo.*

In [6]:
def reune(marcadas):
    """Reúne janelas consecutivas. marcadas: lista de (índice, início, fim, valor) em ordem.
    Devolve intervalos com início, fim, valor máximo e número de janelas."""
    grupos = []
    for i, inicio, fim, valor in marcadas:
        if grupos and grupos[-1]['ultima'] == i - 1:
            g = grupos[-1]
            g['fim'], g['ultima'], g['maximo'], g['janelas'] = fim, i, max(g['maximo'], valor), g['janelas'] + 1
        else:
            grupos.append({'inicio': inicio, 'fim': fim, 'ultima': i, 'maximo': valor, 'janelas': 1})
    return grupos


def itens_panns(janelas):
    """Itens do PANNs e rótulos onipresentes ({rótulo: proporção de janelas})."""
    por_rotulo = {}
    for i, j in enumerate(janelas):
        for rotulo, p in j['rotulos'].items():
            if p >= LIMIAR_PANNS:
                por_rotulo.setdefault(rotulo, []).append((i, j['inicio'], j['fim'], p))
    onipresentes = {r: len(v) / len(janelas) for r, v in por_rotulo.items()
                    if len(v) / len(janelas) > PROPORCAO_ONIPRESENTE_PANNS}
    itens = [{'inicio': g['inicio'], 'fim': g['fim'], 'rotulo': r, 'probabilidade_maxima': round(g['maximo'], 2)}
             for r, v in por_rotulo.items() if r not in onipresentes for g in reune(v)]
    return itens, onipresentes


PANNS = {n: itens_panns(d['panns']) for n, d in DADOS.items()}

for n, (itens, onipresentes) in PANNS.items():
    rotulos = pd.Series([i['rotulo'] for i in itens]).value_counts()
    print(f'Faixa {n:02d} · {len(itens)} itens, {len(rotulos)} rótulos · fora dos itens:',
          {r: f'{p:.0%}' for r, p in onipresentes.items()})
    print('   ', ', '.join(f'{r} ({k})' for r, k in rotulos.items()))

Faixa 01 · 41 itens, 12 rótulos · fora dos itens: {'Speech': '84%'}
    Applause (12), Narration, monologue (9), Male speech, man speaking (8), Eruption (3), Radio (2), Vehicle (1), Noise (1), Speech synthesizer (1), Sheep (1), Bleat (1), Machine gun (1), Gunshot, gunfire (1)
Faixa 02 · 21 itens, 3 rótulos · fora dos itens: {'Speech': '85%'}
    Music (18), Static (2), Rain on surface (1)
Faixa 03 · 27 itens, 4 rótulos · fora dos itens: {'Speech': '98%'}
    Narration, monologue (10), Male speech, man speaking (10), Gunshot, gunfire (4), Artillery fire (3)
Faixa 04 · 9 itens, 3 rótulos · fora dos itens: {'Music': '55%'}
    Speech (4), Musical instrument (4), Wood block (1)
Faixa 05 · 126 itens, 26 rótulos · fora dos itens: {'Speech': '60%'}
    Music (29), Male speech, man speaking (17), Narration, monologue (16), Insect (11), Bee, wasp, etc. (10), Fly, housefly (6), Vehicle (5), Plucked string instrument (4), Musical instrument (4), Guitar (3), Owl (3), Singing bowl (2), Hoot (2), Go

**Resultado.** A retirada vale rótulo por rótulo, de modo que, nas faixas em que “Speech” saiu, permanecem “Male speech, man speaking” e “Narration, monologue”, que somam 33 intervalos na faixa 05. Os trechos musicais chegam fragmentados (18 intervalos de “Music” na faixa 02 e 29 na faixa 05), porque a probabilidade oscila em torno do limiar. Os rótulos que a escuta dirigida da fase 4 classificou na categoria 4, sem som correspondente, como “Owl”, “Cattle, bovinae” e “Gunshot, gunfire”, entram sem nenhuma marca, já que o LLM não recebe as escutas, assim como os da categoria 3, como “Sheep”, que nomeava a tosse da faixa 01. Pela regra do grau de apoio, um único rótulo com probabilidade de 0,5 ou mais recebe grau alto, como “Owl” (0,60), o que deixa claro que o grau mede o apoio nas fontes, e não a presença do som.

## **5. Descrições (CLAP)**

*Toma a descrição em primeiro lugar em cada janela e reúne as janelas consecutivas com a mesma descrição. Antes, retira as descrições que ficam entre as três primeiras em mais de um quarto das janelas da faixa, os atratores da fase 4. A lista de atratores é mostrada aqui para conferência, mas não vai para o LLM, porque o desenho não a inclui na entrada.*

In [7]:
def itens_clap(janelas):
    """Itens do CLAP e descrições atratoras ({descrição: proporção de janelas no top-3})."""
    ordens = [sorted(j['semelhancas'], key=j['semelhancas'].get, reverse=True) for j in janelas]
    top = Counter(d for o in ordens for d in o[:TOP_CLAP])
    atratores = {d: c / len(janelas) for d, c in top.items() if c / len(janelas) > PROPORCAO_ATRATOR_CLAP}
    por_descricao = {}
    for i, (j, o) in enumerate(zip(janelas, ordens)):
        if o[0] not in atratores:
            por_descricao.setdefault(o[0], []).append((i, j['inicio'], j['fim'], 0))
    itens = [{'inicio': g['inicio'], 'fim': g['fim'], 'descricao': d}
             for d, v in por_descricao.items() for g in reune(v)]
    return itens, atratores


CLAP = {n: itens_clap(d['clap']) for n, d in DADOS.items()}

for n, (itens, atratores) in CLAP.items():
    descricoes = pd.Series([i['descricao'] for i in itens], dtype=str).value_counts()
    print(f'Faixa {n:02d} · {len(itens)} itens · atratores retirados:',
          {d: f'{p:.0%}' for d, p in sorted(atratores.items(), key=lambda x: -x[1])})
    print('   ', ', '.join(f'{d} ({k})' for d, k in descricoes.items()) or 'nenhuma descrição restante')

Faixa 01 · 2 itens · atratores retirados: {'the surface noise of an old record': '99%', 'a man giving a speech': '68%', 'a skipping record': '45%', 'clicks of a scratched record': '32%', 'batucada': '32%'}
    women singing (1), silence (1)
Faixa 02 · 2 itens · atratores retirados: {'the surface noise of an old record': '100%', 'clicks of a scratched record': '80%', 'a skipping record': '74%'}
    a man giving a speech (2)
Faixa 03 · 3 itens · atratores retirados: {'the surface noise of an old record': '96%', 'a man giving a speech': '91%', 'music': '58%'}
    batucada (1), footsteps (1), silence (1)
Faixa 04 · 1 itens · atratores retirados: {'the surface noise of an old record': '100%', 'clicks of a scratched record': '70%', 'a skipping record': '66%'}
    a man giving a speech (1)
Faixa 05 · 79 itens · atratores retirados: {'surdo': '75%', 'a man giving a speech': '51%', 'music': '48%', 'batucada': '28%'}
    women singing (22), electrical hum (12), wind (11), silence (8), a church b

**Resultado.** Nas faixas 01 a 04, os atratores ocupam o alto da lista em quase todas as janelas e restam de um a três itens por faixa, de modo que, nessas faixas, a entrada se compõe quase só da fala e do PANNs. Como a regra é calculada faixa a faixa, “a man giving a speech” é atratora nas faixas 01, 03 e 05 e permanece como item nas faixas 02 e 04, e “the surface noise of an old record”, atratora nas outras quatro faixas, permanece na faixa 05 como um único item, de 0 a 3 s, que coincide com o ruído registrado na escuta cega (00:00–00:03). Na faixa 05 entram, entre outras, “women singing” (22 intervalos), “wind” (11), “a church bell” (8) e “ocean waves” (1).

## **6. Montagem e gravação**

*Reúne os três blocos num arquivo por faixa, com identificadores em ordem de tempo (`F-001` para fala, `P-001` para PANNs, `C-001` para CLAP). Cada faixa é identificada apenas pelo número e pela duração. A legenda descreve, sem interpretar, o que é cada campo e como os sinais foram calculados, para que o LLM saiba o que está lendo. O número de tokens é uma estimativa grosseira (um token a cada quatro caracteres), e a contagem exata virá da própria API, no caderno 5b.*

In [8]:
LEGENDA = {
    'fala': (
        'Segmentos produzidos pelo modelo de reconhecimento de fala Whisper (large-v3), sem filtro '
        'de ruído, com início e fim em segundos, o texto escrito pelo modelo e três indicadores do '
        'próprio modelo. avg_logprob é a média do logaritmo da probabilidade dos tokens do texto '
        '(valores mais baixos correspondem a menor confiança do modelo no texto). no_speech_prob é '
        'a probabilidade, estimada pelo modelo, de que o segmento não contenha fala. '
        'compression_ratio é a razão de compressão do texto (valores altos correspondem a texto '
        'repetitivo dentro do segmento). O campo sinais lista os sinais calculados a partir desses dados.'
    ),
    'sinais': {
        'repetição': 'o mesmo texto aparece em três ou mais segmentos da faixa',
        'indicação de ausência de fala': 'no_speech_prob acima de 0,6',
        'baixa confiança': 'avg_logprob abaixo de -1,0',
        'divergência entre transcrições': (
            'menos da metade das palavras do segmento aparece, no mesmo intervalo, numa segunda '
            'transcrição da mesma gravação feita pelo mesmo modelo depois de um filtro de redução de ruído'
        ),
    },
    'eventos_panns': (
        'Intervalos em que o detector de eventos sonoros PANNs (Cnn14, treinado com as 527 categorias '
        'do AudioSet) atribuiu probabilidade de pelo menos 0,3 a um rótulo, em janelas de 2 segundos '
        'com passo de 1 segundo. Janelas consecutivas com o mesmo rótulo foram reunidas num intervalo. '
        'probabilidade_maxima é o maior valor do rótulo no intervalo.'
    ),
    'descricoes_clap': (
        'Intervalos em que uma descrição textual, em inglês, ficou em primeiro lugar entre 28 na '
        'comparação feita pelo modelo CLAP entre o áudio e as descrições, em janelas de 2 segundos '
        'com passo de 1 segundo. Janelas consecutivas com a mesma descrição foram reunidas num '
        'intervalo. As descrições que ficaram entre as três primeiras em mais de um quarto das '
        'janelas da faixa foram retiradas.'
    ),
}


def identifica(itens, prefixo):
    """Ordena por tempo e acrescenta identificadores (F-001, P-001, C-001)."""
    ordenados = sorted(itens, key=lambda x: (x['inicio'], x['fim']))
    return [{'id': f'{prefixo}-{k:03d}', **x} for k, x in enumerate(ordenados, 1)]


def monta_entrada(faixa, duracao, segmentos_a, palavras_c, janelas_panns, janelas_clap):
    """Arquivo de entrada de uma faixa (seção 3 do desenho)."""
    sinais = sinais_fala(segmentos_a, palavras_c)
    fala = [{'inicio': round(s['start'], 2), 'fim': round(s['end'], 2), 'texto': s['text'].strip(),
             'avg_logprob': round(s['avg_logprob'], 2), 'no_speech_prob': round(s['no_speech_prob'], 2),
             'compression_ratio': round(s['compression_ratio'], 2), 'sinais': si}
            for s, si in zip(segmentos_a, sinais)]
    panns, onipresentes = itens_panns(janelas_panns)
    clap, atratores = itens_clap(janelas_clap)
    if onipresentes:
        lista = ', '.join(f'{r} ({p:.0%} das janelas)' for r, p in sorted(onipresentes.items()))
        resumo = f'Rótulos apontados em mais da metade das janelas da faixa, que não entram como itens: {lista}.'
    else:
        resumo = 'Nenhum rótulo foi apontado em mais da metade das janelas da faixa.'
    entrada = {
        'faixa': faixa,
        'duracao_s': round(duracao, 2),
        'legenda': LEGENDA,
        'fala': identifica(fala, 'F'),
        'eventos_panns': identifica(panns, 'P'),
        'resumo_panns': resumo,
        'descricoes_clap': identifica(clap, 'C'),
    }
    return entrada, atratores


ENTRADAS = {}
for n, d in DADOS.items():
    ENTRADAS[n], _ = monta_entrada(n, d['duracao'], d['segmentos_a'], d['palavras_c'], d['panns'], d['clap'])
    with open(os.path.join(PASTA_ENTRADAS, f'faixa{n:02d}.json'), 'w', encoding='utf-8') as f:
        json.dump(ENTRADAS[n], f, ensure_ascii=False, indent=1)

tamanhos = []
for n, e in ENTRADAS.items():
    texto = json.dumps(e, ensure_ascii=False)
    tamanhos.append({'faixa': n, 'fala': len(e['fala']), 'PANNs': len(e['eventos_panns']),
                     'CLAP': len(e['descricoes_clap']), 'caracteres': len(texto), 'tokens (estimativa)': len(texto) // 4})
print(sorted(os.listdir(PASTA_ENTRADAS)))
pd.DataFrame(tamanhos).set_index('faixa')

['faixa01.json', 'faixa02.json', 'faixa03.json', 'faixa04.json', 'faixa05.json']


,fala,PANNs,CLAP,caracteres,tokens (estimativa)
faixa,,,,,
1,63,41,2,19435,4858
2,40,21,2,13452,3363
3,62,27,3,18910,4727
4,8,9,1,4594,1148
5,23,126,79,26065,6516


**Resultado.** Os cinco arquivos foram gravados em `resultados/fase5/entradas`. Somadas, as entradas das cinco faixas têm cerca de 20 mil tokens pela estimativa, perto de 60 mil nas quinze chamadas do caderno 5b. A pesquisadora leu a legenda antes da execução e a manteve sem alterações.

## **7. Conferência das entradas**

*Confere que cada arquivo tem só os campos previstos, que os identificadores são únicos e que todos os intervalos ficam dentro da duração da faixa. Depois mostra, em ordem de tempo e com as três fontes juntas, o primeiro minuto da faixa 05, que é o que o LLM vai ler.*

In [9]:
CAMPOS = {'faixa', 'duracao_s', 'legenda', 'fala', 'eventos_panns', 'resumo_panns', 'descricoes_clap'}
for n, e in ENTRADAS.items():
    itens = e['fala'] + e['eventos_panns'] + e['descricoes_clap']
    ids = [i['id'] for i in itens]
    print(f'Faixa {n:02d} · campos previstos: {set(e) == CAMPOS} · identificadores únicos: {len(ids) == len(set(ids))}'
          f' · intervalos dentro da duração: {all(0 <= i["inicio"] <= i["fim"] <= e["duracao_s"] + 1 for i in itens)}')
    print('   ', e['resumo_panns'])

Faixa 01 · campos previstos: True · identificadores únicos: True · intervalos dentro da duração: True
    Rótulos apontados em mais da metade das janelas da faixa, que não entram como itens: Speech (84% das janelas).
Faixa 02 · campos previstos: True · identificadores únicos: True · intervalos dentro da duração: True
    Rótulos apontados em mais da metade das janelas da faixa, que não entram como itens: Speech (85% das janelas).
Faixa 03 · campos previstos: True · identificadores únicos: True · intervalos dentro da duração: True
    Rótulos apontados em mais da metade das janelas da faixa, que não entram como itens: Speech (98% das janelas).
Faixa 04 · campos previstos: True · identificadores únicos: True · intervalos dentro da duração: True
    Rótulos apontados em mais da metade das janelas da faixa, que não entram como itens: Music (55% das janelas).
Faixa 05 · campos previstos: True · identificadores únicos: True · intervalos dentro da duração: True
    Rótulos apontados em mais d

In [10]:
e = ENTRADAS[5]
itens = e['fala'] + e['eventos_panns'] + e['descricoes_clap']
for i in sorted(itens, key=lambda x: (x['inicio'], x['id'])):
    if i['inicio'] >= 60:
        break
    if i['id'].startswith('F'):
        conteudo = f"“{i['texto']}” · sinais: {', '.join(i['sinais']) or 'nenhum'}"
    elif i['id'].startswith('P'):
        conteudo = f"{i['rotulo']} · {i['probabilidade_maxima']}"
    else:
        conteudo = i['descricao']
    print(f"{i['id']}  {i['inicio']:6.1f} a {i['fim']:6.1f} s  {conteudo}")

C-001     0.0 a    3.0 s  the surface noise of an old record
C-002     2.0 a    4.0 s  silence
C-003     3.0 a    7.0 s  electrical hum
C-004     6.0 a    9.0 s  a skipping record
C-005     9.0 a   12.0 s  electrical hum
C-006    12.0 a   14.0 s  a skipping record
P-001    12.0 a   15.0 s  Music · 0.48
C-007    16.0 a   18.0 s  a church bell
P-002    18.0 a   20.0 s  Singing bowl · 0.35
P-003    18.0 a   21.0 s  Music · 0.6
P-004    21.0 a   26.0 s  Music · 0.88
F-001    25.0 a   30.0 s  “A CIDADE NO BRASIL” · sinais: repetição, indicação de ausência de fala, divergência entre transcrições
P-005    26.0 a   32.0 s  Music · 0.89
P-006    27.0 a   29.0 s  Musical instrument · 0.35
C-008    28.0 a   30.0 s  a church bell
P-007    30.0 a   32.0 s  Singing bowl · 0.34
P-008    32.0 a   62.0 s  Music · 0.86
P-009    36.0 a   38.0 s  Musical instrument · 0.33
C-009    37.0 a   39.0 s  a church bell
C-010    39.0 a   42.0 s  a church bell
C-011    42.0 a   44.0 s  a church bell
C-012    48.0 a

**Resultado.** Os cinco arquivos têm apenas os campos previstos, identificadores únicos e intervalos dentro da duração da faixa. No primeiro minuto da faixa 05, o primeiro segmento do loop (F-001) aparece entre rótulos de música do PANNs e descrições de sino do CLAP, com os sinais de repetição, de indicação de ausência de fala e de divergência.

## **8. Ambiente de execução**

In [11]:
import sys, importlib.metadata as md
from datetime import datetime
print(datetime.now().strftime('%d/%m/%Y %H:%M'))
print('Python', sys.version.split()[0])
print('pandas', md.version('pandas'))

02/11/2026 14:06
Python 3.13.18
pandas 2.2.3


## **9. Testes automáticos**

*Executa todos os testes de `tests/`, os das fases 3 e 4 e os de `tests/test_entradas.py`, que conferem, com casos de resposta conhecida, os quatro sinais da fala, a reunião das janelas, a retirada dos rótulos onipresentes e dos atratores, e a ordem dos identificadores. As funções testadas estão em `src/entradas.py`, cópia idêntica das definidas neste caderno. A biblioteca jiwer é instalada antes, porque os testes da fase 3 dependem dela.*

In [12]:
!pip install -q jiwer==4.0.0
!cd /content/drive/MyDrive/escuta-cerrada-poc && python -m pytest -q tests

...........................                                              [100%]
27 passed in 5.24s


**Resultado.** Os 27 testes passaram (02/11/2026).